<a href="https://colab.research.google.com/github/mdfarhan825301-ship-it/Ecommerce-Analytics-Python-SQL/blob/main/Ecommerce_Problem_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Olist E-Commerce — 15 Question Analysis
### Basic, Intermediate & Advanced SQL-style Problems (solved in Python/Pandas)

**Author:** Md Farhan

This notebook works through 15 specific business questions on the Olist e-commerce dataset,
grouped into Basic, Intermediate, and Advanced tiers. It reuses the same load/clean/merge
pattern as the main EDA notebook so results line up with that analysis.


## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 150)


## 2. Load Data

**If running in Google Colab:** run the upload cell below and select all 7 CSVs when prompted.

**If running locally:** place the CSVs in the same folder as this notebook (or edit `DATA_DIR`)
and skip the upload cell.

Expected filenames: `customers.csv`, `geolocation.csv`, `order_items.csv`, `orders.csv`,
`payments.csv`, `products.csv`, `sellers.csv`


In [ ]:
DATA_DIR = "."  # <-- edit if your CSVs live somewhere else

FILES = {
    "customers":   "/content/customers (2).csv",
    "geolocation": "/content/geolocation.csv",
    "order_items": "/content/order_items.csv",
    "orders":      "/content/orders.csv.csv",
    "payments":    "/content/payments.csv",
    "products":    "/content/products.csv",
    "sellers":     "/content/sellers.csv",
}

import os

def load_csv(key):
    path = os.path.join(DATA_DIR, FILES[key])
    return pd.read_csv(path)

customers   = load_csv("customers")
geolocation = load_csv("geolocation")
order_items = load_csv("order_items")
orders      = load_csv("orders")
payments    = load_csv("payments")
products    = load_csv("products")
sellers     = load_csv("sellers")

# Parse the timestamp columns up front, everything downstream depends on them
date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
for c in date_cols:
    if c in orders.columns:
        orders[c] = pd.to_datetime(orders[c], errors="coerce")

print("Loaded:", {k: v.shape for k, v in
      [('customers', customers), ('orders', orders), ('order_items', order_items),
       ('payments', payments), ('products', products), ('sellers', sellers),
       ('geolocation', geolocation)]})

Loaded: {'customers': (99441, 5), 'orders': (99441, 8), 'order_items': (112650, 7), 'payments': (103886, 5), 'products': (32951, 9), 'sellers': (3095, 4), 'geolocation': (1000163, 5)}


In [ ]:
# Fix the product category column name if it has a literal space (matches source file quirk)
if "product category" in products.columns:
    products = products.rename(columns={"product category": "product_category"})


## 3. Build the Master Table

One row per order item (the finest grain we need), with customer/product/seller attributes
attached. Payments stay separate since they live at the order grain, not the item grain.


In [ ]:
master = (
    order_items
    .merge(orders, on="order_id", how="left")
    .merge(customers, on="customer_id", how="left")
    .merge(products, on="product_id", how="left")
    .merge(sellers, on="seller_id", how="left")
)

master["item_revenue"] = master["price"]
master["order_year"] = master["order_purchase_timestamp"].dt.year
master["order_month"] = master["order_purchase_timestamp"].dt.month
master["order_year_month"] = master["order_purchase_timestamp"].dt.to_period("M")

print("Master table shape:", master.shape)
assert len(master) == len(order_items), "Row count changed during merge — check for fan-out joins"
master.head(3)


Master table shape: (112650, 33)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,product category,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,item_revenue,order_year,order_month,order_year_month
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19,58.9,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ,Cool Stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,27277,volta redonda,SP,58.9,2017,9,2017-09
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03,239.9,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP,pet Shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,3471,sao paulo,SP,239.9,2017,4,2017-04
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18,199.0,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG,Furniture Decoration,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,37564,borda da mata,MG,199.0,2018,1,2018-01


In [ ]:
# Payments aggregated to order grain — needed for the installment question
payments_per_order = payments.groupby("order_id", as_index=False).agg(
    total_payment_value=("payment_value", "sum"),
    payment_installments_max=("payment_installments", "max"),
    n_payment_methods=("payment_type", "nunique"),
)

# Order-level revenue table — used repeatedly in the Advanced section
order_level = (
    master.groupby("order_id", as_index=False)
    .agg(
        order_value=("item_revenue", "sum"),
        customer_unique_id=("customer_unique_id", "first"),
        customer_city=("customer_city", "first"),
        customer_state=("customer_state", "first"),
        order_purchase_timestamp=("order_purchase_timestamp", "first"),
        n_items=("order_item_id", "count"),
    )
)
order_level["order_year"] = order_level["order_purchase_timestamp"].dt.year
order_level["order_month"] = order_level["order_purchase_timestamp"].dt.month
order_level["order_year_month"] = order_level["order_purchase_timestamp"].dt.to_period("M")

print("Order-level table shape:", order_level.shape)
order_level.head(3)


Order-level table shape: (98666, 10)


,order_id,order_value,customer_unique_id,customer_city,customer_state,order_purchase_timestamp,n_items,order_year,order_month,order_year_month
0,00010242fe8c5a6d1ba2dd792cb16214,58.9,871766c5855e863f6eccc05f988b23cb,campos dos goytacazes,RJ,2017-09-13 08:59:02,1,2017,9,2017-09
1,00018f77f2f0320c557190d7a144bdd3,239.9,eb28e67c4c0b83846050ddfb8a35d051,santa fe do sul,SP,2017-04-26 10:53:06,1,2017,4,2017-04
2,000229ec398224ef6ca0657da4fc703e,199.0,3818d81c6709e39d06b2738a8d3a2474,para de minas,MG,2018-01-14 14:33:31,1,2018,1,2018-01


---
# BASIC PROBLEMS
*Objective: Get familiar with the data.*


### B1. Unique cities where customers are located

In [ ]:
unique_cities = customers["customer_city"].dropna().unique()
print(f"Number of unique customer cities: {len(unique_cities)}")
pd.Series(sorted(unique_cities)).to_frame("customer_city")


Number of unique customer cities: 4119


,customer_city
0,abadia dos dourados
1,abadiania
2,abaete
3,abaetetuba
4,abaiara
...,...
4114,xinguara
4115,xique-xique
4116,zacarias
4117,ze doca


### B2. Number of orders placed in 2017

In [ ]:
orders_2017 = orders[orders["order_purchase_timestamp"].dt.year == 2017]
print(f"Orders placed in 2017: {len(orders_2017):,}")


Orders placed in 2017: 45,101


### B3. Total sales per category

In [ ]:
sales_per_category = (
    master.groupby("product category")["item_revenue"]
    .sum()
    .sort_values(ascending=False)
    .rename("total_sales")
)
sales_per_category.to_frame()

,total_sales
product category,
HEALTH BEAUTY,1258681.34
Watches present,1205005.68
bed table bath,1036988.68
sport leisure,988048.97
computer accessories,911954.32
...,...
flowers,1110.04
House Comfort 2,760.27
cds music dvds,730.00


### B4. Percentage of orders paid in installments

In [ ]:
orders_with_payment = orders.merge(payments_per_order, on="order_id", how="left")
pct_installments = (orders_with_payment["payment_installments_max"] > 1).mean() * 100
print(f"Orders paid in installments (>1 installment): {pct_installments:.2f}%")


Orders paid in installments (>1 installment): 51.46%


### B5. Count of customers from each state

In [ ]:
import pandas as pd

# 1. Read the FULL file
customers = pd.read_csv('/content/customers (2).csv')

# 2. Check if it's the correct file
print("Shape:", customers.shape)

Shape: (99441, 5)


In [ ]:
customers_per_state_all = (
    customers.groupby("customer_state")["customer_unique_id"]
    .count()  # this counts every row, duplicates included
    .sort_values(ascending=False)
    .rename("total_records")
    .reset_index()
)

customers_per_state_all.head()

,customer_state,total_records
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045


---
# INTERMEDIATE PROBLEMS
*Objective: Dive deeper into sales and order trends.*


### I1. Number of orders per month in 2018

In [ ]:
orders_2018 = orders[orders["order_purchase_timestamp"].dt.year == 2018].copy()
orders_2018["order_month"] = orders_2018["order_purchase_timestamp"].dt.month

orders_per_month_2018 = (
    orders_2018.groupby("order_month")["order_id"]
    .nunique()
    .rename("n_orders")
    .reindex(range(1, 13))  # keep all 12 months even if some have zero orders
)
orders_per_month_2018.to_frame()


,n_orders
order_month,
1,7269.0
2,6728.0
3,7211.0
4,6939.0
5,6873.0
6,6167.0
7,6292.0
8,6512.0
9,16.0


### I2. Average number of products per order, grouped by customer city

In [ ]:
avg_products_per_order_by_city = (
    order_level.groupby("customer_city")["n_items"]
    .mean()
    .sort_values(ascending=False)
    .rename("avg_products_per_order")
)
avg_products_per_order_by_city.to_frame().head(20)


,avg_products_per_order
customer_city,
padre carvalho,7.00
celso ramos,6.50
datas,6.00
candido godoi,6.00
matias olimpio,5.00
morro de sao paulo,4.00
picarra,4.00
curralinho,4.00
cidelandia,4.00


### I3. Percentage of total revenue contributed by each product category

In [ ]:
total_revenue = master["item_revenue"].sum()

revenue_pct_by_category = (
    (sales_per_category / total_revenue * 100)
    .sort_values(ascending=False)
    .rename("pct_of_total_revenue")
)
revenue_pct_by_category.to_frame().head(10)


,pct_of_total_revenue
product category,
HEALTH BEAUTY,9.260700
Watches present,8.865783
bed table bath,7.629605
sport leisure,7.269533
computer accessories,6.709669
Furniture Decoration,5.369200
Cool Stuff,4.674128
housewares,4.651745
automotive,4.360916


### I4. Correlation between product price and number of times purchased

In [ ]:
product_stats = (
    master.groupby("product_id")
    .agg(
        avg_price=("price", "mean"),
        times_purchased=("order_id", "count"),
    )
)

price_purchase_corr = product_stats["avg_price"].corr(product_stats["times_purchased"])
print(f"Correlation between product price and purchase count: {price_purchase_corr:.4f}")
print("(Negative = cheaper products tend to sell more often; near-zero = little to no linear relationship.)")


Correlation between product price and purchase count: -0.0321
(Negative = cheaper products tend to sell more often; near-zero = little to no linear relationship.)


### I5. Total revenue by seller, ranked

In [ ]:
revenue_by_seller = (
    master.groupby("seller_id")["item_revenue"]
    .sum()
    .sort_values(ascending=False)
    .rename("total_revenue")
    .to_frame()
)
revenue_by_seller["rank"] = revenue_by_seller["total_revenue"].rank(ascending=False, method="dense").astype(int)
revenue_by_seller.head(10)


,total_revenue,rank
seller_id,,
4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1
53243585a1d6dc2643021fd1853d8905,222776.05,2
4a3ca9315b744ce9f8e9374361493884,200472.92,3
fa1c13f2614d7b5c4749cbc52fecda94,194042.03,4
7c67e1448b00f6e969d365cea6b010ab,187923.89,5
7e93a43ef30c4f03f38b393420bc753a,176431.87,6
da8622b14eb17ae2831f4ac5b9dab84a,160236.57,7
7a67c85e85bb2ce8582c35f2203ad736,141745.53,8
1025f0e2d44d7041d6cf58b6550e0bfa,138968.55,9


---
# ADVANCED PROBLEMS
*Objective: Generate strategic and customer-centric insights.*


### A1. Moving average of order values for each customer over their order history

In [ ]:
print(orders.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


In [ ]:
# 1. Keep only delivered orders - this is the big one
orders_delivered = orders[orders['order_status'] == 'delivered']

# 2. Merge: orders + customers + items
df = orders_delivered.merge(customers[['customer_id', 'customer_unique_id']], on='customer_id', how='left')
df = df.merge(order_items[['order_id', 'price', 'freight_value']], on='order_id', how='left')

# 3. Aggregate: order_value = price + freight. This matches most SQL queries
order_values = df.groupby(['order_id', 'customer_unique_id', 'order_purchase_timestamp'])[['price', 'freight_value']].sum().reset_index()
order_values['order_value'] = (order_values['price'] + order_values['freight_value']).round(2)

# 4. Sort + Moving avg
order_values['order_purchase_timestamp'] = pd.to_datetime(order_values['order_purchase_timestamp'])
order_values = order_values.sort_values(['customer_unique_id', 'order_purchase_timestamp'])

order_values['moving_avg_order_value'] = (
    order_values.groupby('customer_unique_id')['order_value']
    .transform(lambda s: s.expanding().mean().round(2))
)

order_values[['order_id', 'customer_unique_id', 'order_purchase_timestamp', 'order_value', 'moving_avg_order_value']].head(10)

,order_id,customer_unique_id,order_purchase_timestamp,order_value,moving_avg_order_value
85426,e22acc9c116caa3f2b7121bbb380d08e,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,141.90,141.90
20134,3594e05a005ac4d06a72673270ef9ec9,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,27.19,27.19
67434,b33ec3b699337181488304f362a6b734,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,86.22,86.22
24483,41272756ecddd9a9ed0180413cc22fb6,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,43.62,43.62
82023,d957021f1127559cd947b62533f484f7,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,196.89,196.89
23425,3e470077b690ea3e3d501cffb5e0c499,0004bd2a26a76fe21f786e4fbd80607f,2018-04-05 19:33:16,166.98,166.98
78476,d0028facea13f508e880202d7097a5a1,00050ab1314c0e55a6ca13cf7181fecf,2018-04-20 12:57:23,35.38,35.38
25945,44e608f2db00c74a1fe329de44416a4e,00053a61a98854899e70ed204dd4bafe,2018-02-28 11:15:41,419.18,419.18
65567,ae76bef74b97bcb0b3e355e60d9a6f9c,0005e1862207bf6ccc02e4228effd9a0,2017-03-04 23:32:12,150.12,150.12
629,01b330808c5819a6a3cb79b72f0b8288,0005ef4cd20d2893f0d9fbd94d3c0d97,2018-03-12 15:22:12,129.76,129.76


### A2. Cumulative sales per month for each year

In [ ]:
monthly_sales = (
    master.groupby(["order_year", "order_month"])["item_revenue"]
    .sum()
    .reset_index()
    .sort_values(["order_year", "order_month"])
)

monthly_sales["cumulative_sales"] = (
    monthly_sales.groupby("order_year")["item_revenue"].cumsum()
)

monthly_sales.rename(columns={"item_revenue": "monthly_sales"})


,order_year,order_month,monthly_sales,cumulative_sales
0,2016,9,267.36,267.36
1,2016,10,49507.66,49775.02
2,2016,12,10.90,49785.92
3,2017,1,120312.87,120312.87
4,2017,2,247303.02,367615.89
5,2017,3,374344.30,741960.19
6,2017,4,359927.23,1101887.42
7,2017,5,506071.14,1607958.56
8,2017,6,433038.60,2040997.16
9,2017,7,498031.48,2539028.64


### A3. Year-over-year growth rate of total sales

In [ ]:
yearly_sales = master.groupby("order_year")["item_revenue"].sum().sort_index()

yoy_growth = (yearly_sales.pct_change() * 100).rename("yoy_growth_pct")

pd.concat([yearly_sales.rename("total_sales"), yoy_growth], axis=1)


,total_sales,yoy_growth_pct
order_year,,
2016,49785.92,NaN
2017,6155806.98,12264.554035
2018,7386050.80,19.985094


### A4. Customer retention rate
Defined as: the percentage of customers who place another order within 6 months of their
first purchase.


In [ ]:
orders = pd.read_csv('/content/orders.csv.csv')
print("Rows in orders.csv:", len(orders))

Rows in orders.csv: 99441


In [ ]:
import pandas as pd
import numpy as np
from dateutil.relativedelta import relativedelta

orders = pd.read_csv('/content/orders.csv.csv')
customers = pd.read_csv('/content/customers (2).csv')

df = orders.merge(customers[['customer_id', 'customer_unique_id']], on='customer_id', how='left')

df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'], errors='coerce')
df = df.dropna(subset=['order_purchase_timestamp'])


first_orders = (
    df.groupby('customer_unique_id')['order_purchase_timestamp']
    .min()
    .reset_index()
    .rename(columns={'order_purchase_timestamp': 'first_purchase'})
)

total_customers = first_orders['customer_unique_id'].nunique()

df2 = df.merge(first_orders, on='customer_unique_id')

# calendar 6-month cutoff -- matches MySQL DATE_ADD(first_purchase, INTERVAL 6 MONTH)
df2['cutoff_6m'] = df2['first_purchase'].apply(lambda d: d + relativedelta(months=6))


df2['is_retained'] = (
    (df2['order_purchase_timestamp'] > df2['first_purchase']) &
    (df2['order_purchase_timestamp'] <= df2['cutoff_6m'])
)


retained_customers = df2.loc[df2['is_retained'], 'customer_unique_id'].nunique()
retention_rate_pct = round(100.0 * retained_customers / total_customers, 2)

print("total_customers:", total_customers)
print("retained_customers:", retained_customers)
print("retention_rate_pct:", retention_rate_pct)

total_customers: 96096
retained_customers: 2234
retention_rate_pct: 2.32


### A5. Top 3 customers who spent the most money in each year

In [ ]:
spend_by_customer_year = (
    order_level.groupby(["order_year", "customer_unique_id"])["order_value"]
    .sum()
    .reset_index()
)

# Rank customers within each year by spend, descending
spend_by_customer_year["rank"] = (
    spend_by_customer_year
    .groupby("order_year")["order_value"]
    .rank(method="min", ascending=False)
)

top3_per_year = (
    spend_by_customer_year[spend_by_customer_year["rank"] <= 3]
    .sort_values(["order_year", "rank"])
    .reset_index(drop=True)
)

top3_per_year


,order_year,customer_unique_id,order_value,rank
0,2016,fdaa290acb9eeacb66fa7f979baa6803,1399.00,1.0
1,2016,753bc5d6efa9e49a03e34cf521a9e124,1299.99,2.0
2,2016,b92a2e5e8a6eabcc80882c7d68b2c70b,1199.00,3.0
3,2017,0a0a92112bd4c708ca5fde585afaa872,13440.00,1.0
4,2017,da122df9eeddfedc1dc1f5349a1a690c,7388.00,2.0
5,2017,dc4802a71eae9be1dd28f5d788ceb526,6735.00,3.0
6,2018,763c8b1c9c68a0229c42c9fc6f662b93,7160.00,1.0
7,2018,459bef486812aa25204be022145caa62,6729.00,2.0
8,2018,5d0a2980b292d049061542014e8960bf,4599.90,3.0
